# Week 6: Izhikevich Model — Quantitative Validation Against Izhikevich (2003)

Reproduces three canonical firing patterns from Izhikevich, E.M. (2003),
"Simple Model of Spiking Neurons," using the exact parameter sets from that paper,
and validates each pattern quantitatively (not just "looks plausible") —
see `tests/test_izhikevich.py` for the corresponding automated checks.

In [ ]:
import importlib.util
if importlib.util.find_spec("snnkit") is None:
    !git clone https://github.com/YOUR_USERNAME/snnkit.git
    %cd snnkit
    !pip install -e . --quiet

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from snnkit.reproducibility import set_seed, get_package_versions
from snnkit.models.izhikevich import (
    REGULAR_SPIKING, INTRINSICALLY_BURSTING, CHATTERING, FAST_SPIKING,
    simulate_izhikevich, spike_times_ms, inter_spike_intervals,
)

set_seed(0)
print(get_package_versions())

In [ ]:
def run(params, I_val, T=1000, v0=-70.0):
    i_trace = jnp.full((T,), I_val)
    v, u, spikes = simulate_izhikevich(i_trace, params, v0=v0)
    st = spike_times_ms(spikes, params.dt)
    isi = inter_spike_intervals(st)
    return v, spikes, st, isi

patterns = {
    "Regular Spiking (RS)": (REGULAR_SPIKING, 10.0),
    "Chattering / Bursting (CH)": (CHATTERING, 10.0),
    "Fast Spiking (FS)": (FAST_SPIKING, 10.0),
}

results = {name: run(params, I_val) for name, (params, I_val) in patterns.items()}

## Three labeled plots: membrane potential traces

Compare these directly against the published Izhikevich (2003) Figure 1/2 traces for the same parameter sets.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
t_ms = np.arange(1000)
for ax, (name, (v, spikes, st, isi)) in zip(axes, results.items()):
    ax.plot(t_ms, v, linewidth=0.8)
    ax.set_ylabel("v (mV)")
    ax.set_title(f"{name}  —  n_spikes={len(st)}")
axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.savefig("izhikevich_three_patterns.png", dpi=150)
plt.show()

## Quantitative validation (not just visual)

In [ ]:
# Regular spiking: spike-frequency adaptation (first ISI shorter than steady-state)
v, spikes, st, isi = results["Regular Spiking (RS)"]
first_isi, steady_isi = isi[0], np.mean(isi[-5:])
print(f"RS: first ISI={first_isi:.1f}ms, steady-state ISI={steady_isi:.1f}ms "
      f"(adaptation ratio={first_isi/steady_isi:.2f}, expect < 1)")
assert first_isi < steady_isi

# Chattering: periodic bursts -- bimodal ISI distribution (short within-burst, long between-burst)
v, spikes, st, isi = results["Chattering / Bursting (CH)"]
short_isis = isi[isi < 15]
long_isis = isi[isi >= 15]
print(f"CH: {len(short_isis)} within-burst ISIs (<15ms), {len(long_isis)} inter-burst gaps (>=15ms), "
      f"inter-burst gap CV={np.std(long_isis)/np.mean(long_isis):.3f}")
assert len(short_isis) > 0 and len(long_isis) >= 3

# Fast spiking: sustained high rate, minimal adaptation
v_rs, _, st_rs, isi_rs = results["Regular Spiking (RS)"]
v_fs, _, st_fs, isi_fs = results["Fast Spiking (FS)"]
print(f"FS: n_spikes={len(st_fs)} vs RS: n_spikes={len(st_rs)} (FS should fire more)")
assert len(st_fs) > len(st_rs)

print("\nAll three patterns pass quantitative validation against their defining characteristics.")

## Parameter sets used (Izhikevich 2003)

| Pattern | a | b | c | d |
|---|---|---|---|---|
| Regular Spiking | 0.02 | 0.2 | -65 | 8 |
| Chattering | 0.02 | 0.2 | -50 | 2 |
| Fast Spiking | 0.1 | 0.2 | -65 | 2 |

All driven with a constant step current I=10 (arbitrary units matching the original paper's convention), dt=1ms, v0=-70mV.